# Semantic Correspondence — Colab pipeline

**Training and orchestration only.** Once the pipeline finishes, download `runs/` and `checkpoints/` from Drive to your local repo and run [`AML_Analysis.ipynb`](AML_Analysis.ipynb) to produce paper-ready tables, plots, and qualitative figures.

Hardware-adaptive: precision, `torch.compile`, `num_workers`, and `resume_save_interval` are picked from the GPU's compute capability and VRAM (no GPU model name is hard-coded).

**Resume:** re-run all cells. Auto-detect uses Drive state: if `runs/pipeline_state.json` or any `checkpoints/*.pt` exists on Drive, it resumes; otherwise it cold-starts.

## 1. GPU probe

Runtime → Change runtime type → **GPU**. Any CUDA GPU works.

In [ ]:
import subprocess
import sys

result = subprocess.run(
    [sys.executable, "-c",
     "import torch;\nprint(torch.__version__);\nprint('cuda_available=', torch.cuda.is_available());\nprint('torch_cuda=', torch.version.cuda)"],
    capture_output=True, text=True,
)
print(result.stdout)

need_reinstall = ("cuda_available= False" in result.stdout) and ("torch_cuda= None" in result.stdout)
if need_reinstall:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "--upgrade", "--force-reinstall",
         "torch", "torchvision", "--index-url", "https://download.pytorch.org/whl/cu121"],
        check=True,
    )

import torch

print("torch", torch.__version__)
print("cuda_available=", torch.cuda.is_available())
print("torch_cuda=", torch.version.cuda)

GPU_NAME = ""
CUDA_MAJOR = 0
CUDA_MINOR = 0
VRAM_GB = 0.0

if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    GPU_NAME = torch.cuda.get_device_name(0)
    CUDA_MAJOR, CUDA_MINOR = p.major, p.minor
    VRAM_GB = p.total_memory / (1024**3)
    print(f"gpu_name={GPU_NAME!r}  capability={CUDA_MAJOR}.{CUDA_MINOR}  total_mem_GB={VRAM_GB:.1f}")
else:
    print("WARNING: no CUDA GPU detected. The Colab notebook is intended for CUDA runtimes.")

if CUDA_MAJOR >= 8 and torch.cuda.is_bf16_supported():
    PRECISION = "auto"
elif CUDA_MAJOR == 7:
    PRECISION = "fp16"
else:
    PRECISION = "fp32"

COMPILE = (CUDA_MAJOR >= 8)

if CUDA_MAJOR >= 8 and VRAM_GB >= 32:
    SPEED_TIER = "fast"
elif CUDA_MAJOR >= 7 and VRAM_GB >= 12:
    SPEED_TIER = "medium"
else:
    SPEED_TIER = "slow"

print(f"capability={CUDA_MAJOR}.{CUDA_MINOR}  vram_gb={VRAM_GB:.1f}  speed_tier={SPEED_TIER}  "
      f"precision={PRECISION}  compile={COMPILE}")


## 2. Ephemeral cleanup

Removes `/content/Semantic-Correspondence` only (not Drive).

In [ ]:
import shutil
from pathlib import Path

FORCE_CLEAN = True
for p in (Path("/content/Semantic-Correspondence"), Path("/content/sample_data")):
    if FORCE_CLEAN and p.exists():
        print("Removing:", p)
        shutil.rmtree(p, ignore_errors=True)


## 3. Clone repository

In [ ]:
import os

REPO_DIR = Path("/content/Semantic-Correspondence")
REPO_URL = "https://github.com/lucaosti/Semantic-Correspondence.git"

if not REPO_DIR.is_dir():
    if REPO_DIR.exists():
        shutil.rmtree(REPO_DIR, ignore_errors=True)
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)], check=True)
else:
    print("Repo exists:", REPO_DIR)

os.chdir(REPO_DIR)
print("REPO =", REPO_DIR.resolve())


## 4. Drive: persist `runs/` and `checkpoints/`

In [ ]:
from google.colab import drive

drive.mount("/content/drive", force_remount=True)

BASE_DIR = Path("/content/drive/MyDrive/Colab Notebooks/AML_results")
RUNS_DIR = BASE_DIR / "runs"
CKPT_DIR = BASE_DIR / "checkpoints"
RUNS_DIR.mkdir(parents=True, exist_ok=True)
CKPT_DIR.mkdir(parents=True, exist_ok=True)

os.chdir(REPO_DIR)
for link_name, target in (("runs", RUNS_DIR), ("checkpoints", CKPT_DIR)):
    p = REPO_DIR / link_name
    if p.is_symlink() or p.is_file():
        p.unlink()
    elif p.is_dir():
        shutil.rmtree(p)
    os.symlink(str(target), str(p))
    print(f"Linked {p} -> {target}")


## 5. SPair-71k dataset

In [ ]:
import tarfile
import urllib.request

os.chdir(REPO_DIR)

SPAIR_URL = "https://cvlab.postech.ac.kr/research/SPair-71k/data/SPair-71k.tar.gz"
DATA_DIR = REPO_DIR / "data"
SPAIR_ROOT = DATA_DIR / "SPair-71k"
TAR_PATH = DATA_DIR / "SPair-71k.tar.gz"
DATA_DIR.mkdir(parents=True, exist_ok=True)

if not SPAIR_ROOT.is_dir():
    if not TAR_PATH.is_file():
        print("Downloading:", SPAIR_URL)
        urllib.request.urlretrieve(SPAIR_URL, TAR_PATH)
    print("Extracting to:", DATA_DIR)
    with tarfile.open(TAR_PATH, "r:gz") as tar:
        if sys.version_info >= (3, 12):
            tar.extractall(path=DATA_DIR, filter="data")
        else:
            tar.extractall(path=DATA_DIR)

print("SPAIR_ROOT =", SPAIR_ROOT, "| exists =", SPAIR_ROOT.is_dir())


## 6. Install package

In [ ]:
os.chdir(REPO_DIR)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", ".[notebook]"], check=True)
os.environ["PYTHONPATH"] = str(REPO_DIR) + os.pathsep + os.environ.get("PYTHONPATH", "")
print("OK:", REPO_DIR)


## 7. Pretrained weights

In [ ]:
os.chdir(REPO_DIR)
subprocess.run([sys.executable, "scripts/download_pretrained_weights.py"], check=True)


## 8. Pipeline configuration

Hardware-adaptive defaults driven by the capability/VRAM detected above:

- **Batch sizes**: PDF defaults (`DINO=20`, `SAM=4`) on every CUDA GPU. Hyperparameters are fixed; only runtime knobs adapt.
- **Precision**: `auto` (bf16) on Ampere+; `fp16` on Volta/Turing; `fp32` fallback.
- **`torch.compile`**: enabled only on Ampere+.
- **`num_workers`**: scaled to the VM (typically 4-8 vCPUs).
- **`resume_save_interval`**: adaptive to `SPEED_TIER`.

**Resume vs cold start:** the next cell sets `START_FROM_SCRATCH` automatically. If Drive already contains `runs/pipeline_state.json` or any `checkpoints/*.pt`, the pipeline **resumes**. Delete those on Drive for a cold start.

In [ ]:
import yaml

os.chdir(REPO_DIR)

DRIVE_AML_BASE = Path("/content/drive/MyDrive/Colab Notebooks/AML_results")
CHECKPOINT_DIR_ON_DRIVE = str(DRIVE_AML_BASE / "checkpoints")
_drive_state = DRIVE_AML_BASE / "runs" / "pipeline_state.json"
_drive_ckpts = DRIVE_AML_BASE / "checkpoints"
_has_resume_context = _drive_state.is_file() or (_drive_ckpts.is_dir() and any(_drive_ckpts.glob("*.pt")))
START_FROM_SCRATCH = not _has_resume_context
print("Pipeline:", "resume" if _has_resume_context else "cold start",
      "| START_FROM_SCRATCH =", START_FROM_SCRATCH)

FT_BATCH_SIZE_BY_BACKBONE = {"dinov2_vitb14": 20, "dinov3_vitb16": 20, "sam_vit_b": 4}
LORA_BATCH_SIZE_BY_BACKBONE = {"dinov2_vitb14": 20, "dinov3_vitb16": 20, "sam_vit_b": 4}
FT_LR = 5e-5
FT_WEIGHT_DECAY = 0.01
LORA_LR = 1e-3
LORA_ALPHA = 16.0
LORA_RANK = 8
LORA_LAST_BLOCKS = 2
LAST_BLOCKS_LIST = [1, 2, 4]
FT_EPOCHS = 50
FT_PATIENCE = 7
LORA_EPOCHS = 50
LORA_PATIENCE = 7

NUM_WORKERS = max(2, min(8, (os.cpu_count() or 4) // 2))
RESUME_SAVE_INTERVAL = {"slow": 50, "medium": 200, "fast": 500}[SPEED_TIER]

if 0 < VRAM_GB < 16:
    print(f"WARNING: VRAM={VRAM_GB:.0f} GB < 16 GB. Batch=20 at 518x518 may OOM; halve batches manually if needed.")

cfg = {
    "paths": {
        "spair_root": str(REPO_DIR / "data" / "SPair-71k"),
        "checkpoint_dir": CHECKPOINT_DIR_ON_DRIVE,
    },
    "runtime": {
        "device": "cuda",
        "precision": PRECISION,
        "compile": COMPILE,
        "num_workers": NUM_WORKERS,
        "preprocess": "FIXED_RESIZE",
        "image_size_by_backbone": {
            "dinov2_vitb14": [518, 518],
            "dinov3_vitb16": [512, 512],
            "sam_vit_b": [512, 512],
        },
        "limit_pairs": 0,
        "eval_split": "test",
        "alphas": [0.05, 0.1, 0.2],
        "wsa_window": 5,
        "wsa_temperature": 1.0,
        "log_batch_interval": 25,
        "resume_save_interval": RESUME_SAVE_INTERVAL,
    },
    "finetune": {
        "last_blocks": LAST_BLOCKS_LIST,
        "epochs": FT_EPOCHS,
        "patience": FT_PATIENCE,
        "batch_size": 20,
        "batch_size_by_backbone": FT_BATCH_SIZE_BY_BACKBONE,
        "lr": FT_LR,
        "weight_decay": FT_WEIGHT_DECAY,
        "dinov2_weights": str(REPO_DIR / "checkpoints" / "dinov2_vitb14_pretrain.pth"),
        "dinov3_weights": str(REPO_DIR / "checkpoints" / "dinov3_vitb16_pretrain_lvd1689m-73cec8be.pth"),
        "sam_checkpoint": str(REPO_DIR / "checkpoints" / "sam_vit_b_01ec64.pth"),
    },
    "lora": {
        "rank": LORA_RANK,
        "alpha": LORA_ALPHA,
        "lr": LORA_LR,
        "last_blocks": LORA_LAST_BLOCKS,
        "epochs": LORA_EPOCHS,
        "patience": LORA_PATIENCE,
        "batch_size": 20,
        "batch_size_by_backbone": LORA_BATCH_SIZE_BY_BACKBONE,
    },
    "workflow_toggles": {
        "run_verify_dataset": True,
        "train_finetune": [True, True, True],
        "train_lora": [True, True, True],
        "run_eval_baseline": [True, True, True],
        "run_eval_baseline_wsa": [True, True, True],
        "run_eval_finetuned_checkpoint": [True, True, True],
        "run_eval_lora_checkpoint": [True, True, True],
        "run_eval_finetuned_wsa": [True, True, True],
        "run_eval_lora_wsa": [True, True, True],
        "run_export_metrics_tables": True,
        "run_pytest": False,
        "pipeline_resume": not START_FROM_SCRATCH,
    },
}

cfg_path = REPO_DIR / "config.yaml"
with open(cfg_path, "w", encoding="utf-8") as f:
    yaml.safe_dump(cfg, f, sort_keys=False, allow_unicode=True, default_flow_style=False)

print("Written:", cfg_path)
print(f"speed_tier={SPEED_TIER}  precision={PRECISION}  compile={COMPILE}  "
      f"num_workers={NUM_WORKERS}  resume_save_interval={RESUME_SAVE_INTERVAL}")
print("checkpoint_dir:", cfg["paths"]["checkpoint_dir"])
print("finetune batch map:", cfg["finetune"]["batch_size_by_backbone"])
print("lora batch map:", cfg["lora"]["batch_size_by_backbone"])


## 9. Run pipeline (live dashboard)

In [ ]:
import json
import threading
from collections import deque

from IPython.display import clear_output

os.chdir(REPO_DIR)

env = dict(os.environ)
if START_FROM_SCRATCH:
    env["SEMANTIC_CORRESPONDENCE_PIPELINE_RESET"] = "1"
else:
    env.pop("SEMANTIC_CORRESPONDENCE_PIPELINE_RESET", None)
env.pop("SEMANTIC_CORRESPONDENCE_PIPELINE_LOG_FILE_ONLY", None)
env["PYTHONUNBUFFERED"] = "1"
_pp = env.get("PYTHONPATH", "")
env["PYTHONPATH"] = f"{REPO_DIR}{os.pathsep}{_pp}" if _pp else str(REPO_DIR)

STAGE_EVENTS_PATH = REPO_DIR / "runs" / "logs" / "stage_events.jsonl"
TAIL_LINES = 40
REFRESH_SECONDS = 3.0
cmd = [sys.executable, "-u", "scripts/run_pipeline.py", "--config", "config.yaml"]

_output_lines: deque = deque(maxlen=TAIL_LINES)
_proc_done = threading.Event()
_return_code: list = []


def _stream_subprocess() -> None:
    proc = subprocess.Popen(
        cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, bufsize=1, env=env, cwd=str(REPO_DIR),
    )
    assert proc.stdout is not None
    for line in proc.stdout:
        _output_lines.append(line.rstrip())
    _return_code.append(proc.wait())
    _proc_done.set()


threading.Thread(target=_stream_subprocess, daemon=True).start()


def _read_stage_events() -> list:
    if not STAGE_EVENTS_PATH.is_file():
        return []
    events = []
    with open(STAGE_EVENTS_PATH, "r", encoding="utf-8") as f:
        for raw in f:
            raw = raw.strip()
            if not raw:
                continue
            try:
                events.append(json.loads(raw))
            except json.JSONDecodeError:
                pass
    return events


def _render(events: list) -> None:
    done = [e["stage_id"] for e in events if e.get("action") == "done"]
    skipped = [e["stage_id"] for e in events if e.get("action") == "skip"]
    failed = [e["stage_id"] for e in events if e.get("action") == "fail"]
    latest_start = next((e for e in reversed(events) if e.get("action") == "start"), None)
    current = latest_start["stage_id"] if latest_start else "(waiting...)"
    rc = _return_code[0] if _return_code else None
    if not _proc_done.is_set():
        status = "RUNNING"
    elif rc == 0:
        status = "COMPLETED"
    else:
        status = f"FAILED (exit={rc})"
    sep = "=" * 64
    print(sep)
    print(f"  Pipeline: {status}")
    print(f"  Current stage: {current}")
    print(f"  Completed: {len(done):3d}  Skipped: {len(skipped):3d}  Failed: {len(failed):3d}")
    if done:
        tail = done[-6:]
        suffix = "..." if len(done) > 6 else ""
        print(f"  Last done: {', '.join(tail)}{suffix}")
    if failed:
        print(f"  FAILED: {', '.join(failed)}")
    print(sep)
    print(f"--- last {TAIL_LINES} log lines ---")
    for line in _output_lines:
        print(line)


while not _proc_done.is_set():
    clear_output(wait=True)
    _render(_read_stage_events())
    _proc_done.wait(timeout=REFRESH_SECONDS)

clear_output(wait=True)
_render(_read_stage_events())

rc = _return_code[0] if _return_code else -1
if rc != 0:
    raise subprocess.CalledProcessError(rc, cmd)
print("\nPipeline completed successfully.")


## 10. Stage summary (sanity check)

In [ ]:
import pandas as pd

p = REPO_DIR / "runs" / "logs" / "stage_events.jsonl"
if not p.is_file():
    print("No file:", p)
else:
    events = []
    with open(p, "r", encoding="utf-8") as f:
        for raw in f:
            raw = raw.strip()
            if raw:
                try:
                    events.append(json.loads(raw))
                except json.JSONDecodeError:
                    pass
    done = [e["stage_id"] for e in events if e.get("action") == "done"]
    skipped = [e["stage_id"] for e in events if e.get("action") == "skip"]
    failed = [e["stage_id"] for e in events if e.get("action") == "fail"]
    print(f"events={len(events)} done={len(done)} skipped={len(skipped)} failed={len(failed)}")
    if failed:
        print("Failed:", failed)
    display(pd.DataFrame(events).tail(20))


---

**Pipeline completed.**

**Next step (analysis):** download `runs/pipeline_exports/` and `checkpoints/` from Drive (`MyDrive/Colab Notebooks/AML_results/`) into your local repo (under `runs/` and `checkpoints/` respectively), then open [`AML_Analysis.ipynb`](AML_Analysis.ipynb) locally to produce paper-ready tables, plots, and qualitative figures.